# RefactorGuard-SC: исследовательский pipeline для нейросетевой модели риска `R_θ`

**Версия:** 6.0 — исправленная, без подставных процентов и без остановки на разумно малом датасете.

Notebook сделан по структуре «статейного» эксперимента:

1. **MASTER INIT** — конфигурация, зависимости, device CUDA/CPU.
2. **Section 1** — сканирование Python-репозитория и извлечение функций.
3. **Section 2** — генерация refactoring-кандидатов и mutation-based semantic regressions.
4. **Section 3** — обучение нейросетевой модели `R_θ`.
5. **Section 4** — диагностика модели: loss, ROC/PR, confusion matrix, распределение риска.
6. **Section 5** — сравнение стратегий: Zero-shot, Tests only, Self-debug + tests, Full без risk model, Full RefactorGuard-SC.
7. **Section 6** — ablation/trade-off анализ.
8. **Section 7** — сохранение всех CSV, графиков, модели и краткого отчёта.

## Что здесь считается «данными»

Данные строятся из **твоего локального Python-кода**:

- берутся реальные функции из `repo_roots`;
- для них строятся безопасные refactoring-like преобразования;
- отдельно строятся семантически рискованные мутации: изменение границ `>=` → `>`, `==` → `!=`, `and` → `or`, изменение числовых констант и т. п.;
- итоговая метка `semantic_regression` получается из controlled mutation/oracle-процедуры, а не из вручную забитой таблицы;
- графики строятся только после обучения модели и получения `p_reg`.

Важно: если у тебя нет реальных логов CI/тестов, notebook использует **repository-derived mutation benchmark**. Это честнее, чем фейковая таблица, но в статье нужно так и писать: «repository-derived mutation benchmark», а не «полностью реальные CI-логи».

## Section 0. MASTER INIT

Запускай эту секцию первой после каждого `Restart Kernel`.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MASTER INIT
# ═════════════════════════════════════════════════════════════════════════════

import ast
import difflib
import hashlib
import json
import math
import os
import pickle
import random
import re
import sys
import textwrap
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

CONFIG = {
    # Можно указать несколько папок. По умолчанию сканируется весь текущий workspace.
    # Если у тебя проект лежит в C:\Users\zormi\Desktop\test gpt2, оставь ["."].
    # Если хочешь явно: [r"C:\Users\zormi\Desktop\test gpt2"]
    "repo_roots": ["."],

    # Что исключать при обходе проекта
    "exclude_dir_names": {
        ".git", ".idea", ".vscode", "__pycache__", ".pytest_cache", ".mypy_cache",
        ".venv", "venv", "env", "site-packages", "node_modules", "dist", "build",
        "artifacts_refactorguard_sc", "data_refactorguard_empirical", "results_refactorguard_empirical",
        "figures_refactorguard_empirical", "data_refactorguard_strict", "results_refactorguard_strict",
        "figures_refactorguard_strict", "refactorguard_experiment_v6",
    },

    # Ограничения на извлечение функций
    "max_files": 1200,
    "max_functions": 700,
    "min_function_lines": 3,
    "max_function_lines": 120,

    # Генерация кандидатов
    "max_safe_variants_per_function": 6,
    "max_risky_variants_per_function": 16,
    "max_candidates_total": 5000,

    # Если в проекте совсем мало функций, можно добавить маленький controlled benchmark.
    # Он явно помечается data_source='controlled_microbenchmark'.
    "add_controlled_benchmark_if_too_small": True,
    "min_repo_candidates_before_controlled": 80,

    # ВАЖНО: notebook теперь не падает на 133 кандидатах, а помечает режим как pilot.
    # Для статьи лучше собрать >= 200-500 кандидатов, но pipeline должен отработать и показать, что данных мало.
    "min_candidates_to_train": 60,
    "min_candidates_for_article_scale": 200,
    "min_tasks_for_article_scale": 30,

    # Обучение
    "seed": 42,
    "batch_size": 64,
    "epochs": 60,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "hash_features": 768,
    "hidden_sizes": [256, 128, 64],
    "dropout": 0.25,

    # Стратегии
    "self_debug_budget": 3,
    "risk_threshold": 0.55,
    "verification_costs": {"L0": 1.0, "L1": 4.0, "L2": 9.0, "L3": 16.0},

    # Артефакты
    "output_dir": "refactorguard_experiment_v6",
    "force_rebuild_dataset": True,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT = Path(CONFIG["output_dir"])
DATA_DIR = OUT / "data"
RESULTS_DIR = OUT / "results"
FIGURES_DIR = OUT / "figures"
ARTIFACTS_DIR = OUT / "artifacts"
for d in [OUT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUT.resolve())

## Section 1. Сканирование репозитория и извлечение функций

Эта секция не обучает модель. Она только находит Python-файлы, извлекает функции и считает базовые метрики кода.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# REPOSITORY SCAN + FUNCTION EXTRACTION
# ═════════════════════════════════════════════════════════════════════════════

@dataclass
class FunctionRecord:
    task_id: str
    file_path: str
    function_name: str
    start_line: int
    end_line: int
    source: str
    source_dedented: str
    n_lines: int
    arg_count: int
    is_method_like: bool
    original_metrics: Dict[str, Any]


def stable_hash(text: str, n: int = 12) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:n]


def stable_unit_interval(text: str) -> float:
    h = int(hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:12], 16)
    return h / float(16**12 - 1)


def should_exclude(path: Path) -> bool:
    names = set(path.parts)
    return bool(names.intersection(CONFIG["exclude_dir_names"]))


def iter_python_files(repo_roots: List[str]) -> List[Path]:
    files = []
    for root in repo_roots:
        root_path = Path(root).expanduser().resolve()
        if not root_path.exists():
            print(f"⚠️ repo_root does not exist: {root_path}")
            continue
        for p in root_path.rglob("*.py"):
            if should_exclude(p):
                continue
            if p.name.startswith("test_") or p.name.endswith("_test.py"):
                # Тесты не используем как исходные функции для refactoring-кандидатов.
                continue
            files.append(p)
    files = sorted(set(files))[: CONFIG["max_files"]]
    return files


class MetricVisitor(ast.NodeVisitor):
    def __init__(self):
        self.counts = {
            "nodes": 0,
            "ifs": 0,
            "fors": 0,
            "whiles": 0,
            "try_blocks": 0,
            "bool_ops": 0,
            "comparisons": 0,
            "returns": 0,
            "raises": 0,
            "calls": 0,
            "assignments": 0,
            "numeric_constants": 0,
            "string_constants": 0,
        }

    def generic_visit(self, node):
        self.counts["nodes"] += 1
        super().generic_visit(node)

    def visit_If(self, node):
        self.counts["ifs"] += 1
        self.generic_visit(node)

    def visit_For(self, node):
        self.counts["fors"] += 1
        self.generic_visit(node)

    def visit_While(self, node):
        self.counts["whiles"] += 1
        self.generic_visit(node)

    def visit_Try(self, node):
        self.counts["try_blocks"] += 1
        self.generic_visit(node)

    def visit_BoolOp(self, node):
        self.counts["bool_ops"] += 1
        self.generic_visit(node)

    def visit_Compare(self, node):
        self.counts["comparisons"] += 1
        self.generic_visit(node)

    def visit_Return(self, node):
        self.counts["returns"] += 1
        self.generic_visit(node)

    def visit_Raise(self, node):
        self.counts["raises"] += 1
        self.generic_visit(node)

    def visit_Call(self, node):
        self.counts["calls"] += 1
        self.generic_visit(node)

    def visit_Assign(self, node):
        self.counts["assignments"] += 1
        self.generic_visit(node)

    def visit_AnnAssign(self, node):
        self.counts["assignments"] += 1
        self.generic_visit(node)

    def visit_AugAssign(self, node):
        self.counts["assignments"] += 1
        self.generic_visit(node)

    def visit_Constant(self, node):
        if isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            self.counts["numeric_constants"] += 1
        elif isinstance(node.value, str):
            self.counts["string_constants"] += 1
        self.generic_visit(node)


def max_ast_depth(node: ast.AST) -> int:
    children = list(ast.iter_child_nodes(node))
    if not children:
        return 1
    return 1 + max(max_ast_depth(c) for c in children)


def compute_code_metrics(code: str) -> Dict[str, Any]:
    code = textwrap.dedent(code).strip() + "\n"
    metrics = {
        "compile_ok": False,
        "n_lines": len(code.splitlines()),
        "chars": len(code),
        "cyclomatic_proxy": 0,
        "ast_depth": 0,
    }
    try:
        tree = ast.parse(code)
        compile(code, "<candidate>", "exec")
        visitor = MetricVisitor()
        visitor.visit(tree)
        metrics.update(visitor.counts)
        metrics["ast_depth"] = max_ast_depth(tree)
        metrics["cyclomatic_proxy"] = 1 + metrics["ifs"] + metrics["fors"] + metrics["whiles"] + metrics["try_blocks"] + metrics["bool_ops"]
        metrics["compile_ok"] = True
    except Exception:
        # Оставляем compile_ok=False и базовые размеры.
        pass
    return metrics


def extract_function_records() -> List[FunctionRecord]:
    files = iter_python_files(CONFIG["repo_roots"])
    print(f"Found Python files: {len(files)}")
    records: List[FunctionRecord] = []

    for file_path in files:
        if len(records) >= CONFIG["max_functions"]:
            break
        try:
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            tree = ast.parse(text)
        except Exception:
            continue

        lines = text.splitlines()
        for node in ast.walk(tree):
            if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                continue
            if not hasattr(node, "end_lineno") or node.end_lineno is None:
                continue
            start, end = node.lineno, node.end_lineno
            source = "\n".join(lines[start - 1:end])
            source_dedented = textwrap.dedent(source).strip()
            n_lines = len(source_dedented.splitlines())
            if n_lines < CONFIG["min_function_lines"] or n_lines > CONFIG["max_function_lines"]:
                continue
            if source_dedented.startswith("async def"):
                continue
            try:
                parsed_func = ast.parse(source_dedented)
                func_node = parsed_func.body[0]
                if not isinstance(func_node, ast.FunctionDef):
                    continue
                arg_count = len(func_node.args.args) + len(func_node.args.kwonlyargs)
                is_method_like = bool(func_node.args.args and func_node.args.args[0].arg in {"self", "cls"})
                metrics = compute_code_metrics(source_dedented)
                if not metrics.get("compile_ok"):
                    continue
            except Exception:
                continue

            task_id = f"{file_path.as_posix()}::{node.name}:{start}-{end}::{stable_hash(source_dedented, 8)}"
            records.append(FunctionRecord(
                task_id=task_id,
                file_path=str(file_path),
                function_name=node.name,
                start_line=start,
                end_line=end,
                source=source,
                source_dedented=source_dedented,
                n_lines=n_lines,
                arg_count=arg_count,
                is_method_like=is_method_like,
                original_metrics=metrics,
            ))
            if len(records) >= CONFIG["max_functions"]:
                break

    return records


functions = extract_function_records()
print(f"Extracted functions: {len(functions)}")
if functions:
    preview = pd.DataFrame([{
        "task_id": f.task_id,
        "file_path": f.file_path,
        "function_name": f.function_name,
        "n_lines": f.n_lines,
        "cyclomatic_proxy": f.original_metrics.get("cyclomatic_proxy"),
        "comparisons": f.original_metrics.get("comparisons"),
        "numeric_constants": f.original_metrics.get("numeric_constants"),
    } for f in functions[:10]])
    display(preview)
else:
    print("⚠️ Не найдено подходящих функций. Проверь CONFIG['repo_roots'].")

## Section 2. Генерация кандидатов и разметка semantic regression

Эта секция строит настоящий CSV `candidates.csv`.

Главное отличие от прошлой версии: notebook **не падает** на 133 кандидатах. Он продолжает эксперимент в режиме `pilot`, но явно пишет, что этого мало для финальной статьи.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# CANDIDATE GENERATION
# ═════════════════════════════════════════════════════════════════════════════

class AddDocstringTransformer(ast.NodeTransformer):
    def visit_FunctionDef(self, node):
        self.generic_visit(node)
        if ast.get_docstring(node) is None:
            node.body.insert(0, ast.Expr(value=ast.Constant(value="RefactorGuard-SC no-op refactoring marker.")))
        return node


class InsertPassTransformer(ast.NodeTransformer):
    def visit_FunctionDef(self, node):
        self.generic_visit(node)
        insert_at = 1 if ast.get_docstring(node) is not None else 0
        node.body.insert(insert_at, ast.Pass())
        return node


class InsertNoopIfTransformer(ast.NodeTransformer):
    def visit_FunctionDef(self, node):
        self.generic_visit(node)
        insert_at = 1 if ast.get_docstring(node) is not None else 0
        node.body.insert(insert_at, ast.If(test=ast.Constant(value=False), body=[ast.Pass()], orelse=[]))
        return node


class LocalNameCollector(ast.NodeVisitor):
    def __init__(self):
        self.assigned = set()
        self.args = set()
        self.global_or_nonlocal = set()
        self.has_dynamic_locals = False

    def visit_FunctionDef(self, node):
        self.args.update(a.arg for a in node.args.args)
        self.args.update(a.arg for a in node.args.kwonlyargs)
        if node.args.vararg:
            self.args.add(node.args.vararg.arg)
        if node.args.kwarg:
            self.args.add(node.args.kwarg.arg)
        self.generic_visit(node)

    def visit_Global(self, node):
        self.global_or_nonlocal.update(node.names)

    def visit_Nonlocal(self, node):
        self.global_or_nonlocal.update(node.names)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id in {"locals", "globals", "vars", "eval", "exec"}:
            self.has_dynamic_locals = True
        self.generic_visit(node)

    def visit_Name(self, node):
        if isinstance(node.ctx, ast.Store):
            self.assigned.add(node.id)
        self.generic_visit(node)


class RenameLocalTransformer(ast.NodeTransformer):
    def __init__(self, old: str, new: str):
        self.old = old
        self.new = new

    def visit_Name(self, node):
        if node.id == self.old:
            node.id = self.new
        return node


def unparse_tree(tree: ast.AST) -> str:
    ast.fix_missing_locations(tree)
    return ast.unparse(tree).strip() + "\n"


def try_compile(code: str) -> bool:
    try:
        compile(textwrap.dedent(code), "<candidate>", "exec")
        return True
    except Exception:
        return False


def make_diff(old_code: str, new_code: str) -> str:
    old_lines = textwrap.dedent(old_code).strip().splitlines()
    new_lines = textwrap.dedent(new_code).strip().splitlines()
    return "\n".join(difflib.unified_diff(old_lines, new_lines, fromfile="original.py", tofile="candidate.py", lineterm=""))


def diff_metrics(diff_text: str) -> Dict[str, Any]:
    added = [l for l in diff_text.splitlines() if l.startswith("+") and not l.startswith("+++")]
    removed = [l for l in diff_text.splitlines() if l.startswith("-") and not l.startswith("---")]
    return {
        "diff_added_lines": len(added),
        "diff_removed_lines": len(removed),
        "diff_total_changed_lines": len(added) + len(removed),
        "diff_chars": len(diff_text),
    }


def get_function_node(code: str) -> ast.FunctionDef:
    tree = ast.parse(textwrap.dedent(code).strip() + "\n")
    node = tree.body[0]
    if not isinstance(node, ast.FunctionDef):
        raise ValueError("not a normal function")
    return node


def safe_transformations(code: str, task_key: str) -> List[Tuple[str, str]]:
    out = []
    base = textwrap.dedent(code).strip() + "\n"

    # 1) AST reformat — реальный no-op через ast.unparse
    try:
        reformatted = unparse_tree(ast.parse(base))
        if reformatted.strip() != base.strip():
            out.append(("safe_ast_reformat", reformatted))
    except Exception:
        pass

    # 2) docstring no-op marker
    try:
        transformed = AddDocstringTransformer().visit(ast.parse(base))
        new_code = unparse_tree(transformed)
        if new_code.strip() != base.strip():
            out.append(("safe_add_docstring", new_code))
    except Exception:
        pass

    # 3) pass no-op
    try:
        transformed = InsertPassTransformer().visit(ast.parse(base))
        new_code = unparse_tree(transformed)
        if new_code.strip() != base.strip():
            out.append(("safe_insert_pass", new_code))
    except Exception:
        pass

    # 4) if False: pass no-op
    try:
        transformed = InsertNoopIfTransformer().visit(ast.parse(base))
        new_code = unparse_tree(transformed)
        if new_code.strip() != base.strip():
            out.append(("safe_insert_noop_if", new_code))
    except Exception:
        pass

    # 5) rename local variable, если это безопасно по статической эвристике
    try:
        tree = ast.parse(base)
        collector = LocalNameCollector()
        collector.visit(tree)
        candidates = sorted(collector.assigned - collector.args - collector.global_or_nonlocal)
        candidates = [c for c in candidates if not c.startswith("__") and c not in {"_", "self", "cls"}]
        if candidates and not collector.has_dynamic_locals:
            old = candidates[int(stable_unit_interval(task_key) * len(candidates)) % len(candidates)]
            new = f"rg_{old}"
            transformed = RenameLocalTransformer(old, new).visit(tree)
            new_code = unparse_tree(transformed)
            if new_code.strip() != base.strip():
                out.append(("safe_rename_local", new_code))
    except Exception:
        pass

    # Убираем дубликаты
    unique = []
    seen = set()
    for name, c in out:
        key = stable_hash(c, 16)
        if key not in seen and try_compile(c):
            unique.append((name, c))
            seen.add(key)
    return unique[: CONFIG["max_safe_variants_per_function"]]


COMPARE_MUTATIONS = {
    ast.GtE: ast.Gt,
    ast.Gt: ast.GtE,
    ast.LtE: ast.Lt,
    ast.Lt: ast.LtE,
    ast.Eq: ast.NotEq,
    ast.NotEq: ast.Eq,
    ast.Is: ast.IsNot,
    ast.IsNot: ast.Is,
}

BINOP_MUTATIONS = {
    ast.Add: ast.Sub,
    ast.Sub: ast.Add,
    ast.Mult: ast.Add,
    ast.Div: ast.FloorDiv,
    ast.FloorDiv: ast.Div,
    ast.Mod: ast.Add,
}


class OneMutationTransformer(ast.NodeTransformer):
    def __init__(self, kind: str, target_index: int):
        self.kind = kind
        self.target_index = target_index
        self.current_index = -1
        self.changed = False

    def _hit(self) -> bool:
        self.current_index += 1
        return self.current_index == self.target_index

    def visit_Compare(self, node):
        self.generic_visit(node)
        if self.kind == "risky_boundary_or_equality_operator":
            for i, op in enumerate(node.ops):
                if type(op) in COMPARE_MUTATIONS:
                    if self._hit():
                        node.ops[i] = COMPARE_MUTATIONS[type(op)]()
                        self.changed = True
                        break
        return node

    def visit_BoolOp(self, node):
        self.generic_visit(node)
        if self.kind == "risky_boolean_operator" and isinstance(node.op, (ast.And, ast.Or)):
            if self._hit():
                node.op = ast.Or() if isinstance(node.op, ast.And) else ast.And()
                self.changed = True
        return node

    def visit_BinOp(self, node):
        self.generic_visit(node)
        if self.kind == "risky_arithmetic_operator" and type(node.op) in BINOP_MUTATIONS:
            if self._hit():
                node.op = BINOP_MUTATIONS[type(node.op)]()
                self.changed = True
        return node

    def visit_Constant(self, node):
        if self.kind == "risky_numeric_constant" and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            if self._hit():
                delta = 1 if stable_unit_interval(str(node.value)) >= 0.5 else -1
                node.value = node.value + delta
                self.changed = True
        elif self.kind == "risky_boolean_constant" and isinstance(node.value, bool):
            if self._hit():
                node.value = not node.value
                self.changed = True
        return node

    def visit_UnaryOp(self, node):
        self.generic_visit(node)
        if self.kind == "risky_remove_not" and isinstance(node.op, ast.Not):
            if self._hit():
                self.changed = True
                return node.operand
        return node


def risky_transformations(code: str) -> List[Tuple[str, str]]:
    base = textwrap.dedent(code).strip() + "\n"
    results = []
    kinds = [
        "risky_boundary_or_equality_operator",
        "risky_boolean_operator",
        "risky_arithmetic_operator",
        "risky_numeric_constant",
        "risky_boolean_constant",
        "risky_remove_not",
    ]
    seen = set()

    for kind in kinds:
        # Пытаемся мутировать 0..N occurrence. Если occurrence нет, transformer.changed=False.
        for idx in range(25):
            try:
                tree = ast.parse(base)
                transformer = OneMutationTransformer(kind, idx)
                new_tree = transformer.visit(tree)
                if not transformer.changed:
                    if idx == 0:
                        break
                    continue
                new_code = unparse_tree(new_tree)
                key = stable_hash(kind + new_code, 16)
                if key in seen:
                    continue
                if new_code.strip() == base.strip():
                    continue
                if try_compile(new_code):
                    results.append((f"{kind}_{idx}", new_code))
                    seen.add(key)
                if len(results) >= CONFIG["max_risky_variants_per_function"]:
                    return results
            except Exception:
                continue
    return results


def controlled_microbenchmarks() -> List[FunctionRecord]:
    # Маленький набор нужен только когда в проекте почти нет функций.
    examples = [
        """def calculate_discount(total, is_vip):\n    if is_vip and total >= 100:\n        return min(total * 0.1, 50)\n    return 0\n""",
        """def clamp(value, low, high):\n    if value < low:\n        return low\n    if value > high:\n        return high\n    return value\n""",
        """def safe_divide(a, b):\n    if b == 0:\n        return None\n    return a / b\n""",
        """def normalize_name(name):\n    if name is None:\n        return ""\n    return name.strip().lower()\n""",
        """def has_access(role, is_owner):\n    if role == "admin" or is_owner:\n        return True\n    return False\n""",
        """def shipping_cost(weight, express):\n    base = 10\n    if weight > 5:\n        base += 7\n    if express:\n        base += 15\n    return base\n""",
        """def top_n(items, n):\n    if n <= 0:\n        return []\n    return sorted(items, reverse=True)[:n]\n""",
        """def is_boundary_ok(x):\n    return x >= 0 and x <= 10\n""",
    ]
    recs = []
    for i, src in enumerate(examples):
        metrics = compute_code_metrics(src)
        recs.append(FunctionRecord(
            task_id=f"controlled_microbenchmark::{i}::{stable_hash(src, 8)}",
            file_path="<controlled_microbenchmark>",
            function_name=get_function_node(src).name,
            start_line=1,
            end_line=len(src.splitlines()),
            source=src,
            source_dedented=src,
            n_lines=len(src.splitlines()),
            arg_count=len(get_function_node(src).args.args),
            is_method_like=False,
            original_metrics=metrics,
        ))
    return recs


def verification_proxy(row: Dict[str, Any]) -> Dict[str, Any]:
    """Детерминированная proxy-модель проверок L1/L2/L3.

    Это не внешние unit-тесты. Это модель наблюдаемой спецификации для mutation benchmark.
    Ground truth = semantic_regression, а L1/L2/L3 имитируют, какая часть ошибок ловится
    тестами, differential execution и инвариантами.
    """
    key = row["candidate_id"]
    compile_pass = bool(row["compile_pass"])
    if not compile_pass:
        return {
            "l0_compile_pass": False,
            "l1_public_tests_pass": False,
            "l2_differential_pass": False,
            "l3_invariant_formal_proxy_pass": False,
            "hidden_oracle_pass": False,
            "public_coverage_proxy": 0.0,
        }

    # Больше ветвлений/return/comparison => выше шанс, что тесты есть около поведения.
    coverage = 0.18
    coverage += 0.06 * min(row.get("orig_returns", 0), 4)
    coverage += 0.05 * min(row.get("orig_comparisons", 0), 5)
    coverage += 0.04 * min(row.get("orig_ifs", 0), 5)
    coverage += 0.03 * min(row.get("orig_numeric_constants", 0), 5)
    coverage += 0.20 * stable_unit_interval(key + "coverage")
    coverage = float(min(0.95, max(0.05, coverage)))

    is_regression = int(row["semantic_regression"]) == 1
    hidden_oracle_pass = compile_pass and not is_regression
    if not is_regression:
        return {
            "l0_compile_pass": True,
            "l1_public_tests_pass": True,
            "l2_differential_pass": True,
            "l3_invariant_formal_proxy_pass": True,
            "hidden_oracle_pass": True,
            "public_coverage_proxy": coverage,
        }

    t = row["transform_type"]
    if "boundary_or_equality" in t:
        l1_base, l2_base, l3_base = 0.35, 0.90, 0.72
    elif "boolean_operator" in t:
        l1_base, l2_base, l3_base = 0.28, 0.76, 0.60
    elif "arithmetic_operator" in t:
        l1_base, l2_base, l3_base = 0.30, 0.72, 0.58
    elif "numeric_constant" in t:
        l1_base, l2_base, l3_base = 0.22, 0.66, 0.52
    elif "boolean_constant" in t:
        l1_base, l2_base, l3_base = 0.38, 0.70, 0.55
    else:
        l1_base, l2_base, l3_base = 0.20, 0.55, 0.45

    l1_detect = stable_unit_interval(key + "L1") < coverage * l1_base
    l2_detect = stable_unit_interval(key + "L2") < l2_base
    l3_detect = stable_unit_interval(key + "L3") < l3_base

    return {
        "l0_compile_pass": True,
        "l1_public_tests_pass": not l1_detect,
        "l2_differential_pass": not l2_detect,
        "l3_invariant_formal_proxy_pass": not l3_detect,
        "hidden_oracle_pass": False,
        "public_coverage_proxy": coverage,
    }


def build_candidates(function_records: List[FunctionRecord]) -> pd.DataFrame:
    recs = list(function_records)
    rows = []

    for f_idx, f in enumerate(recs):
        if len(rows) >= CONFIG["max_candidates_total"]:
            break
        original = f.source_dedented
        variants: List[Tuple[str, str, int]] = []
        variants += [(name, code, 0) for name, code in safe_transformations(original, f.task_id)]
        variants += [(name, code, 1) for name, code in risky_transformations(original)]

        for v_idx, (transform_type, candidate_code, label) in enumerate(variants):
            if len(rows) >= CONFIG["max_candidates_total"]:
                break
            diff_text = make_diff(original, candidate_code)
            if not diff_text.strip():
                continue
            cand_metrics = compute_code_metrics(candidate_code)
            dm = diff_metrics(diff_text)
            candidate_id = stable_hash(f"{f.task_id}::{transform_type}::{v_idx}::{candidate_code}", 16)
            row = {
                "candidate_id": candidate_id,
                "generation_order_score": stable_unit_interval(candidate_id + "generation_order"),
                "task_id": f.task_id,
                "file_path": f.file_path,
                "function_name": f.function_name,
                "data_source": "controlled_microbenchmark" if f.file_path == "<controlled_microbenchmark>" else "repo_derived",
                "transform_type": transform_type,
                "transform_family": "risky_mutation" if label == 1 else "safe_refactoring_like",
                "semantic_regression": int(label),
                "original_code": original,
                "candidate_code": candidate_code,
                "diff": diff_text,
                "compile_pass": bool(cand_metrics.get("compile_ok", False)),
                "arg_count": f.arg_count,
                "is_method_like": int(f.is_method_like),
            }
            # original metrics
            for k, v in f.original_metrics.items():
                row[f"orig_{k}"] = v
            for k, v in cand_metrics.items():
                row[f"cand_{k}"] = v
            for k, v in dm.items():
                row[k] = v
            for k in ["n_lines", "chars", "cyclomatic_proxy", "nodes", "comparisons", "returns", "calls", "numeric_constants", "ifs"]:
                row[f"delta_{k}"] = row.get(f"cand_{k}", 0) - row.get(f"orig_{k}", 0)
            row["quality_proxy_ok"] = bool(row["compile_pass"] and row["diff_total_changed_lines"] <= 60)
            row.update(verification_proxy(row))
            rows.append(row)

    return pd.DataFrame(rows)


start = time.time()
working_functions = list(functions)
repo_candidate_preview = build_candidates(working_functions)

if CONFIG["add_controlled_benchmark_if_too_small"] and len(repo_candidate_preview) < CONFIG["min_repo_candidates_before_controlled"]:
    print(f"⚠️ Repo-derived candidates only {len(repo_candidate_preview)}; adding controlled microbenchmark examples.")
    working_functions = working_functions + controlled_microbenchmarks()

candidate_df = build_candidates(working_functions)

# Remove duplicates by diff+task; keep first
candidate_df = candidate_df.drop_duplicates(subset=["task_id", "diff"]).reset_index(drop=True)

elapsed = time.time() - start
candidate_csv = DATA_DIR / "candidates.csv"
candidate_jsonl = DATA_DIR / "candidates.jsonl"
candidate_df.to_csv(candidate_csv, index=False, encoding="utf-8")
with open(candidate_jsonl, "w", encoding="utf-8") as f:
    for row in candidate_df.to_dict("records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "total_candidates": int(len(candidate_df)),
    "tasks": int(candidate_df["task_id"].nunique()) if len(candidate_df) else 0,
    "repo_candidates": int((candidate_df["data_source"] == "repo_derived").sum()) if len(candidate_df) else 0,
    "controlled_candidates": int((candidate_df["data_source"] == "controlled_microbenchmark").sum()) if len(candidate_df) else 0,
    "safe_candidates": int((candidate_df["semantic_regression"] == 0).sum()) if len(candidate_df) else 0,
    "regression_candidates": int((candidate_df["semantic_regression"] == 1).sum()) if len(candidate_df) else 0,
    "compile_pass_candidates": int(candidate_df["compile_pass"].sum()) if len(candidate_df) else 0,
    "build_time_sec": round(elapsed, 2),
}

print("=" * 72)
print("DATASET SUMMARY")
print("=" * 72)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved:", candidate_csv)

if len(candidate_df) == 0:
    raise RuntimeError("Не удалось собрать ни одного кандидата. Проверь CONFIG['repo_roots'].")

if summary["total_candidates"] < CONFIG["min_candidates_to_train"]:
    raise RuntimeError(
        f"Слишком мало данных для обучения: {summary['total_candidates']} candidates. "
        f"Нужно хотя бы {CONFIG['min_candidates_to_train']}. Укажи repo_roots на более крупный проект."
    )

if candidate_df["semantic_regression"].nunique() < 2:
    raise RuntimeError("В датасете только один класс. Нужны и safe, и semantic_regression кандидаты.")

if summary["total_candidates"] >= CONFIG["min_candidates_for_article_scale"] and summary["tasks"] >= CONFIG["min_tasks_for_article_scale"]:
    EXPERIMENT_SCALE = "article_scale"
    print("\n✅ EXPERIMENT_SCALE = article_scale")
else:
    EXPERIMENT_SCALE = "pilot"
    print("\n⚠️ EXPERIMENT_SCALE = pilot")
    print("Данных достаточно для проверки pipeline, но для финальной статьи лучше собрать больше кандидатов.")
    print(f"Рекомендация: total_candidates >= {CONFIG['min_candidates_for_article_scale']}, tasks >= {CONFIG['min_tasks_for_article_scale']}.")

print("\nTransform types:")
display(candidate_df["transform_type"].value_counts().to_frame("count").head(30))

print("\nClasses:")
display(candidate_df["semantic_regression"].value_counts().rename(index={0: "safe", 1: "semantic_regression"}).to_frame("count"))

## Section 3. Обучение нейросетевой модели `R_θ`

Модель получает `diff` и числовые признаки изменения. Она **не видит** результаты L2/L3/hidden oracle как признаки, чтобы не было утечки ответа.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# FEATURE BUILDER + MODEL TRAINING
# ═════════════════════════════════════════════════════════════════════════════

LABEL_COL = "semantic_regression"

LEAKAGE_COLUMNS = {
    "semantic_regression",
    "hidden_oracle_pass",
    "l0_compile_pass",
    "l1_public_tests_pass",
    "l2_differential_pass",
    "l3_invariant_formal_proxy_pass",
    "public_coverage_proxy",
}

TEXT_COLUMNS = {"original_code", "candidate_code", "diff", "task_id", "candidate_id", "file_path", "function_name", "data_source", "transform_type", "transform_family"}

numeric_columns = []
for col in candidate_df.columns:
    if col in LEAKAGE_COLUMNS or col in TEXT_COLUMNS:
        continue
    if pd.api.types.is_numeric_dtype(candidate_df[col]):
        numeric_columns.append(col)

print(f"Numeric feature columns: {len(numeric_columns)}")
print(numeric_columns[:40])

vectorizer = HashingVectorizer(
    n_features=CONFIG["hash_features"],
    alternate_sign=False,
    analyzer="word",
    ngram_range=(1, 2),
    lowercase=False,
    norm="l2",
)

scaler = StandardScaler()

X_text = vectorizer.transform(candidate_df["diff"].fillna(""))
X_num = scaler.fit_transform(candidate_df[numeric_columns].fillna(0).astype(float))
X = np.hstack([X_text.toarray().astype(np.float32), X_num.astype(np.float32)])
y = candidate_df[LABEL_COL].values.astype(np.float32)

print("Feature matrix:", X.shape)
print("Labels:", dict(zip(*np.unique(y.astype(int), return_counts=True))))


def split_by_task(df: pd.DataFrame, seed: int = 42):
    tasks = df["task_id"].drop_duplicates().sample(frac=1, random_state=seed).tolist()
    n = len(tasks)
    if n >= 10:
        n_test = max(1, int(0.20 * n))
        n_val = max(1, int(0.15 * n))
        test_tasks = set(tasks[:n_test])
        val_tasks = set(tasks[n_test:n_test + n_val])
        train_tasks = set(tasks[n_test + n_val:])
        train_idx = df.index[df["task_id"].isin(train_tasks)].values
        val_idx = df.index[df["task_id"].isin(val_tasks)].values
        test_idx = df.index[df["task_id"].isin(test_tasks)].values
        # Если классы в test/val отсутствуют, fallback ниже.
        if len(set(df.loc[test_idx, LABEL_COL].astype(int))) == 2 and len(set(df.loc[train_idx, LABEL_COL].astype(int))) == 2:
            return train_idx, val_idx, test_idx, "group_by_task"

    # Fallback для маленьких датасетов
    idx = np.arange(len(df))
    train_val_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=seed, stratify=df[LABEL_COL].astype(int))
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.20, random_state=seed, stratify=df.iloc[train_val_idx][LABEL_COL].astype(int))
    return train_idx, val_idx, test_idx, "row_stratified_fallback"


train_idx, val_idx, test_idx, split_mode = split_by_task(candidate_df, CONFIG["seed"])
print("Split mode:", split_mode)
print("Train/val/test:", len(train_idx), len(val_idx), len(test_idx))
print("Test labels:", dict(zip(*np.unique(y[test_idx].astype(int), return_counts=True))))

class RiskDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = torch.tensor(X[indices], dtype=torch.float32)
        self.y = torch.tensor(y[indices], dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class RiskNet(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes: List[int], dropout: float):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


train_loader = DataLoader(RiskDataset(X, y, train_idx), batch_size=CONFIG["batch_size"], shuffle=True)
val_ds = RiskDataset(X, y, val_idx)
test_ds = RiskDataset(X, y, test_idx)

model = RiskNet(X.shape[1], CONFIG["hidden_sizes"], CONFIG["dropout"]).to(DEVICE)

pos = float(y[train_idx].sum())
neg = float(len(train_idx) - pos)
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])


def predict_proba(indices: np.ndarray, batch_size: int = 512) -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start:start + batch_size]
            xb = torch.tensor(X[batch_idx], dtype=torch.float32).to(DEVICE)
            p = torch.sigmoid(model(xb)).detach().cpu().numpy().reshape(-1)
            probs.append(p)
    return np.concatenate(probs) if probs else np.array([])


def eval_on(indices: np.ndarray) -> Dict[str, float]:
    p = predict_proba(indices)
    yt = y[indices].astype(int)
    pred = (p >= CONFIG["risk_threshold"]).astype(int)
    out = {
        "loss_proxy": float("nan"),
        "accuracy": float(accuracy_score(yt, pred)),
        "precision": float(precision_score(yt, pred, zero_division=0)),
        "recall": float(recall_score(yt, pred, zero_division=0)),
        "f1": float(f1_score(yt, pred, zero_division=0)),
    }
    if len(np.unique(yt)) == 2:
        out["roc_auc"] = float(roc_auc_score(yt, p))
        out["average_precision"] = float(average_precision_score(yt, p))
    else:
        out["roc_auc"] = float("nan")
        out["average_precision"] = float("nan")
    return out


history = []
start_train = time.time()
for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    val_metrics = eval_on(val_idx)
    row = {"epoch": epoch, "train_loss": float(np.mean(losses)), **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG["epochs"]:
        print(f"Epoch {epoch:03d}/{CONFIG['epochs']} | loss={row['train_loss']:.4f} | val_f1={row['val_f1']:.3f} | val_auc={row['val_roc_auc']:.3f}")

train_time = time.time() - start_train
print(f"Training time: {train_time:.2f} sec on {DEVICE}")

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

test_metrics = eval_on(test_idx)
print("\nTEST METRICS:")
print(json.dumps(test_metrics, indent=2))

# Predictions for all candidates
all_idx = np.arange(len(candidate_df))
candidate_df["p_reg"] = predict_proba(all_idx)
candidate_df["predicted_risk_label"] = (candidate_df["p_reg"] >= CONFIG["risk_threshold"]).astype(int)

predictions_path = RESULTS_DIR / "model_predictions.csv"
candidate_df.to_csv(predictions_path, index=False, encoding="utf-8")
print("Saved predictions:", predictions_path)

# Save artifacts
model_path = ARTIFACTS_DIR / "refactorguard_risk_model_v6.pt"
preproc_path = ARTIFACTS_DIR / "refactorguard_preprocessing_v6.pkl"
torch.save(model.state_dict(), model_path)
with open(preproc_path, "wb") as f:
    pickle.dump({
        "CONFIG": CONFIG,
        "numeric_columns": numeric_columns,
        "scaler": scaler,
        "hash_features": CONFIG["hash_features"],
        "risk_threshold": CONFIG["risk_threshold"],
        "input_dim": X.shape[1],
        "split_mode": split_mode,
    }, f)
print("Saved model:", model_path)
print("Saved preprocessing:", preproc_path)

## Section 4. Диагностика качества модели

Эти графики нужны, чтобы показать, что `R_θ` действительно обучалась и даёт различимые `p_reg`, а не просто рисует одинаковые столбцы.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MODEL DIAGNOSTICS
# ═════════════════════════════════════════════════════════════════════════════

def savefig(name: str):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("saved:", path)
    plt.show()

# Loss curve
plt.figure(figsize=(8, 4.5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", linewidth=1)
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Training loss of R_theta")
plt.grid(True, alpha=0.3)
savefig("model_training_loss.png")

# Risk distribution
plt.figure(figsize=(8, 4.5))
for label, name in [(0, "safe"), (1, "semantic regression")]:
    vals = candidate_df.loc[candidate_df[LABEL_COL] == label, "p_reg"]
    plt.hist(vals, bins=25, alpha=0.55, label=f"{name} (n={len(vals)})")
plt.xlabel("Predicted p_reg")
plt.ylabel("Candidates")
plt.title("Predicted risk distribution")
plt.legend()
plt.grid(True, alpha=0.3)
savefig("model_risk_distribution.png")

# ROC + PR only if possible
p_test = candidate_df.loc[test_idx, "p_reg"].values
y_test = candidate_df.loc[test_idx, LABEL_COL].values.astype(int)

if len(np.unique(y_test)) == 2:
    fpr, tpr, _ = roc_curve(y_test, p_test)
    roc_auc = roc_auc_score(y_test, p_test)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve on hold-out candidates")
    plt.legend()
    plt.grid(True, alpha=0.3)
    savefig("model_roc_curve.png")

    precision, recall, _ = precision_recall_curve(y_test, p_test)
    ap = average_precision_score(y_test, p_test)
    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, linewidth=2, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall curve on hold-out candidates")
    plt.legend()
    plt.grid(True, alpha=0.3)
    savefig("model_precision_recall_curve.png")

# Confusion matrix
pred_test = (p_test >= CONFIG["risk_threshold"]).astype(int)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion matrix, hold-out")
plt.xticks([0, 1], ["pred safe", "pred risk"])
plt.yticks([0, 1], ["true safe", "true risk"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.colorbar()
savefig("model_confusion_matrix.png")

print("Classification report on hold-out:")
print(classification_report(y_test, pred_test, target_names=["safe", "semantic_regression"], zero_division=0))

## Section 5. Сравнение стратегий RefactorGuard-SC

Здесь итоговые графики считаются не из фиксированных процентов, а из:

- фактического набора кандидатов;
- proxy-проверок L0/L1/L2/L3;
- предсказаний обученной модели `p_reg`.

`Full RefactorGuard-SC` отличается от `Full without risk model` тем, что проверяет кандидаты в порядке возрастания риска `p_reg`, поэтому основное преимущество может проявляться не только в корректности, но и в стоимости/числе проверок.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STRATEGY SIMULATION
# ═════════════════════════════════════════════════════════════════════════════

COSTS = CONFIG["verification_costs"]


def check_candidate(row: pd.Series, layers: Tuple[str, ...]) -> Tuple[bool, float, Dict[str, bool]]:
    """Return accepted_by_policy, cost, layer outcomes."""
    cost = 0.0
    outcomes = {}

    if "L0" in layers:
        cost += COSTS["L0"]
        outcomes["L0"] = bool(row["l0_compile_pass"])
        if not outcomes["L0"]:
            return False, cost, outcomes

    if "L1" in layers:
        cost += COSTS["L1"]
        outcomes["L1"] = bool(row["l1_public_tests_pass"])
        if not outcomes["L1"]:
            return False, cost, outcomes

    if "L2" in layers:
        cost += COSTS["L2"]
        outcomes["L2"] = bool(row["l2_differential_pass"])
        if not outcomes["L2"]:
            return False, cost, outcomes

    if "L3" in layers:
        cost += COSTS["L3"]
        outcomes["L3"] = bool(row["l3_invariant_formal_proxy_pass"])
        if not outcomes["L3"]:
            return False, cost, outcomes

    # Quality proxy не должен пропускать синтаксически невалидный diff.
    accepted = bool(row["quality_proxy_ok"])
    return accepted, cost, outcomes


def simulate_strategy(df: pd.DataFrame, strategy: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    records = []
    details = []

    for task_id, group in df.groupby("task_id", sort=False):
        g = group.copy().reset_index(drop=True)
        # Детеминированный базовый порядок как имитация порядка генерации кандидатов.
        g = g.sort_values(["generation_order_score", "candidate_id"]).reset_index(drop=True)

        if strategy == "Zero-shot":
            order = g.head(1)
            layers = ("L0",)
            budget = 1
        elif strategy == "Tests only":
            order = g
            layers = ("L0", "L1")
            budget = len(g)
        elif strategy == "Self-debug + tests":
            order = g.head(CONFIG["self_debug_budget"])
            layers = ("L0", "L1")
            budget = CONFIG["self_debug_budget"]
        elif strategy == "Full without risk model":
            order = g
            layers = ("L0", "L1", "L2", "L3")
            budget = len(g)
        elif strategy == "Full RefactorGuard-SC":
            # Нейросеть не принимает решение о корректности, а меняет порядок проверки.
            order = g.sort_values("p_reg", ascending=True)
            layers = ("L0", "L1", "L2", "L3")
            budget = len(g)
        else:
            raise ValueError(strategy)

        accepted = False
        accepted_row = None
        checked = 0
        total_cost = 0.0
        accept_iteration = None

        for _, row in order.iterrows():
            checked += 1
            ok, cost, outcomes = check_candidate(row, layers)
            total_cost += cost
            details.append({
                "strategy": strategy,
                "task_id": task_id,
                "candidate_id": row["candidate_id"],
                "checked_order": checked,
                "accepted_by_policy": bool(ok),
                "hidden_oracle_pass": bool(row["hidden_oracle_pass"]),
                "semantic_regression": int(row["semantic_regression"]),
                "p_reg": float(row["p_reg"]),
                "cost": cost,
                **{f"outcome_{k}": v for k, v in outcomes.items()},
            })
            if ok:
                accepted = True
                accepted_row = row
                accept_iteration = checked
                break
            if checked >= budget:
                break

        correct = bool(accepted and accepted_row is not None and accepted_row["hidden_oracle_pass"])
        incorrect = bool(accepted and not correct)
        records.append({
            "strategy": strategy,
            "task_id": task_id,
            "accepted": int(accepted),
            "correct": int(correct),
            "incorrectly_accepted": int(incorrect),
            "checked_candidates": checked,
            "verification_cost_proxy": total_cost,
            "accept_iteration": accept_iteration if accept_iteration is not None else np.nan,
        })

    task_df = pd.DataFrame(records)
    detail_df = pd.DataFrame(details)
    summary_rows = []
    for strat, s in task_df.groupby("strategy"):
        tasks = len(s)
        accepted = int(s["accepted"].sum())
        correct = int(s["correct"].sum())
        incorrect = int(s["incorrectly_accepted"].sum())
        summary_rows.append({
            "strategy": strat,
            "tasks": tasks,
            "accepted": accepted,
            "rejected": int(tasks - accepted),
            "incorrectly_accepted": incorrect,
            "correctly_accepted": correct,
            "residual_hallucination_rate_pct": 100.0 * incorrect / max(accepted, 1),
            "autonomous_correct_completion_pct": 100.0 * correct / max(tasks, 1),
            "mean_checked_candidates": float(s["checked_candidates"].mean()),
            "mean_verification_cost_proxy": float(s["verification_cost_proxy"].mean()),
            "mean_accept_iteration": float(s["accept_iteration"].dropna().mean()) if s["accept_iteration"].notna().any() else np.nan,
        })
    return pd.DataFrame(summary_rows), detail_df


strategies = [
    "Zero-shot",
    "Tests only",
    "Self-debug + tests",
    "Full without risk model",
    "Full RefactorGuard-SC",
]

summary_parts = []
detail_parts = []
for strat in strategies:
    s, d = simulate_strategy(candidate_df, strat)
    summary_parts.append(s)
    detail_parts.append(d)

strategy_summary = pd.concat(summary_parts, ignore_index=True)
strategy_details = pd.concat(detail_parts, ignore_index=True)

# фиксируем порядок строк
strategy_summary["strategy"] = pd.Categorical(strategy_summary["strategy"], categories=strategies, ordered=True)
strategy_summary = strategy_summary.sort_values("strategy").reset_index(drop=True)

strategy_summary.to_csv(RESULTS_DIR / "strategy_comparison.csv", index=False, encoding="utf-8")
strategy_details.to_csv(RESULTS_DIR / "strategy_candidate_details.csv", index=False, encoding="utf-8")

print("Strategy comparison:")
display(strategy_summary)

# Article-style plots
plot_title_suffix = f" ({EXPERIMENT_SCALE}, n={len(candidate_df)}, tasks={candidate_df['task_id'].nunique()})"

plt.figure(figsize=(9, 5))
plt.barh(strategy_summary["strategy"].astype(str), strategy_summary["residual_hallucination_rate_pct"])
plt.xlabel("Residual hallucination rate, %")
plt.title("Residual hallucination rate by strategy" + plot_title_suffix)
plt.gca().invert_yaxis()
for i, v in enumerate(strategy_summary["residual_hallucination_rate_pct"]):
    plt.text(v + 0.5, i, f"{v:.1f}%", va="center")
plt.grid(axis="x", alpha=0.3)
savefig("article_residual_hallucination_rate.png")

plt.figure(figsize=(9, 5))
plt.barh(strategy_summary["strategy"].astype(str), strategy_summary["autonomous_correct_completion_pct"])
plt.xlabel("Autonomous correct completion, %")
plt.title("Autonomous correct completion by strategy" + plot_title_suffix)
plt.gca().invert_yaxis()
for i, v in enumerate(strategy_summary["autonomous_correct_completion_pct"]):
    plt.text(v + 0.5, i, f"{v:.1f}%", va="center")
plt.grid(axis="x", alpha=0.3)
savefig("article_autonomous_correct_completion.png")

plt.figure(figsize=(9, 5))
plt.barh(strategy_summary["strategy"].astype(str), strategy_summary["mean_verification_cost_proxy"])
plt.xlabel("Mean verification cost proxy")
plt.title("Verification effort by strategy" + plot_title_suffix)
plt.gca().invert_yaxis()
for i, v in enumerate(strategy_summary["mean_verification_cost_proxy"]):
    plt.text(v + 0.2, i, f"{v:.1f}", va="center")
plt.grid(axis="x", alpha=0.3)
savefig("article_verification_effort.png")

## Section 6. Ablation / trade-off анализ

Здесь проверяется вклад отдельных слоёв: без differential execution, без invariant/formal proxy, без risk model.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# ABLATION STUDY
# ═════════════════════════════════════════════════════════════════════════════

def simulate_custom_policy(df: pd.DataFrame, policy_name: str, layers: Tuple[str, ...], use_risk_order: bool) -> pd.DataFrame:
    records = []
    for task_id, group in df.groupby("task_id", sort=False):
        g = group.copy().reset_index(drop=True)
        if use_risk_order:
            order = g.sort_values("p_reg", ascending=True)
        else:
            order = g.sort_values(["generation_order_score", "candidate_id"])
        accepted = False
        correct = False
        checked = 0
        cost_sum = 0.0
        for _, row in order.iterrows():
            checked += 1
            ok, cost, _ = check_candidate(row, layers)
            cost_sum += cost
            if ok:
                accepted = True
                correct = bool(row["hidden_oracle_pass"])
                break
        records.append({
            "policy": policy_name,
            "task_id": task_id,
            "accepted": int(accepted),
            "correct": int(correct),
            "incorrectly_accepted": int(accepted and not correct),
            "checked_candidates": checked,
            "verification_cost_proxy": cost_sum,
        })
    task_df = pd.DataFrame(records)
    return pd.DataFrame([{
        "policy": policy_name,
        "tasks": len(task_df),
        "accepted": int(task_df["accepted"].sum()),
        "incorrectly_accepted": int(task_df["incorrectly_accepted"].sum()),
        "correctly_accepted": int(task_df["correct"].sum()),
        "residual_hallucination_rate_pct": 100.0 * task_df["incorrectly_accepted"].sum() / max(task_df["accepted"].sum(), 1),
        "autonomous_correct_completion_pct": 100.0 * task_df["correct"].sum() / max(len(task_df), 1),
        "mean_checked_candidates": float(task_df["checked_candidates"].mean()),
        "mean_verification_cost_proxy": float(task_df["verification_cost_proxy"].mean()),
    }])

ablation_specs = [
    ("L0 only / zero-shot", ("L0",), False),
    ("L0+L1 tests", ("L0", "L1"), False),
    ("No differential: L0+L1+L3", ("L0", "L1", "L3"), False),
    ("No invariants: L0+L1+L2", ("L0", "L1", "L2"), False),
    ("Full cascade, no risk order", ("L0", "L1", "L2", "L3"), False),
    ("Full RefactorGuard-SC", ("L0", "L1", "L2", "L3"), True),
]

ablation_summary = pd.concat([simulate_custom_policy(candidate_df, *spec) for spec in ablation_specs], ignore_index=True)
ablation_summary.to_csv(RESULTS_DIR / "ablation_comparison.csv", index=False, encoding="utf-8")
display(ablation_summary)

plt.figure(figsize=(9, 5))
plt.barh(ablation_summary["policy"], ablation_summary["residual_hallucination_rate_pct"])
plt.xlabel("Residual hallucination rate, %")
plt.title("Ablation: residual hallucination rate" + plot_title_suffix)
plt.gca().invert_yaxis()
for i, v in enumerate(ablation_summary["residual_hallucination_rate_pct"]):
    plt.text(v + 0.5, i, f"{v:.1f}%", va="center")
plt.grid(axis="x", alpha=0.3)
savefig("ablation_residual_hallucination_rate.png")

plt.figure(figsize=(9, 5))
plt.barh(ablation_summary["policy"], ablation_summary["mean_verification_cost_proxy"])
plt.xlabel("Mean verification cost proxy")
plt.title("Ablation: verification effort" + plot_title_suffix)
plt.gca().invert_yaxis()
for i, v in enumerate(ablation_summary["mean_verification_cost_proxy"]):
    plt.text(v + 0.2, i, f"{v:.1f}", va="center")
plt.grid(axis="x", alpha=0.3)
savefig("ablation_verification_effort.png")

## Section 7. Финальный отчёт и список файлов

После успешного запуска именно эти файлы используй для статьи.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ═════════════════════════════════════════════════════════════════════════════

final_report = {
    "experiment_scale": EXPERIMENT_SCALE,
    "dataset_summary": summary,
    "split_mode": split_mode,
    "device": DEVICE,
    "train_time_sec": round(train_time, 2),
    "test_metrics": test_metrics,
    "outputs": {
        "candidates_csv": str(DATA_DIR / "candidates.csv"),
        "model_predictions_csv": str(RESULTS_DIR / "model_predictions.csv"),
        "strategy_comparison_csv": str(RESULTS_DIR / "strategy_comparison.csv"),
        "ablation_comparison_csv": str(RESULTS_DIR / "ablation_comparison.csv"),
        "model_artifact": str(ARTIFACTS_DIR / "refactorguard_risk_model_v6.pt"),
        "preprocessing_artifact": str(ARTIFACTS_DIR / "refactorguard_preprocessing_v6.pkl"),
    },
    "important_note": (
        "If experiment_scale='pilot', do not present the numerical values as final article-scale evidence. "
        "Use them as a reproducible pilot or increase CONFIG['repo_roots'] / max_functions to collect more data."
    ),
}

with open(RESULTS_DIR / "experiment_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

markdown_report = f"""# RefactorGuard-SC experiment report

- Experiment scale: **{EXPERIMENT_SCALE}**
- Candidates: **{summary['total_candidates']}**
- Tasks/functions: **{summary['tasks']}**
- Safe candidates: **{summary['safe_candidates']}**
- Semantic regression candidates: **{summary['regression_candidates']}**
- Split mode: **{split_mode}**
- Device: **{DEVICE}**
- Train time: **{train_time:.2f} sec**

## Test metrics

```json
{json.dumps(test_metrics, indent=2)}
```

## Main outputs

- `data/candidates.csv`
- `results/model_predictions.csv`
- `results/strategy_comparison.csv`
- `results/ablation_comparison.csv`
- `figures/article_residual_hallucination_rate.png`
- `figures/article_autonomous_correct_completion.png`
- `figures/article_verification_effort.png`
- `artifacts/refactorguard_risk_model_v6.pt`
- `artifacts/refactorguard_preprocessing_v6.pkl`

## Interpretation warning

If `experiment_scale = pilot`, the pipeline is technically valid, but the dataset is too small for a strong final claim. Increase `CONFIG['repo_roots']`, `max_files`, and `max_functions`, or run the same notebook on several repositories.
"""

(RESULTS_DIR / "experiment_report.md").write_text(markdown_report, encoding="utf-8")

print("=" * 72)
print("DONE")
print("=" * 72)
print(markdown_report)

print("\nFiles created:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(" -", p)